# OATS — Analytic Agency Horizon (E1–E7)

**Operational Agency Time-Scaling** — the closed-form Gaussian channel model
underlying Experiments E1–E7 of *Forecasting Without Power* (Clayton, 2026).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DrRalphClayton/fawp-index/blob/main/notebooks/oats_demo.ipynb)

---

**What this notebook covers:**

| Section | Content |
|---|---|
| 1 | The analytic model — I(τ) and τ_h in one line |
| 2 | E1: Parameter sweep — analytic vs Monte Carlo horizons |
| 3 | E2: Convergence — MI estimator follows 1/√N |
| 4 | E3: Noise-law discrimination — linear vs quadratic vs saturating |
| 5 | E4: Distributional robustness — Gaussian vs Uniform vs Student-t |
| 6 | E5: Control cliff — MI decay → stability failure |
| 7 | E6: Agency capacity surfaces — quantization cliff + erasure slope |
| 8 | E7: Quantum checks — Bell/CHSH + no-signaling |
| 9 | Plain-English diagnosis with explain() |

> Ralph Clayton (2026) · [doi:10.5281/zenodo.18663547](https://doi.org/10.5281/zenodo.18663547) · [Book](https://www.amazon.com/dp/B0GS1ZVNM7/)

In [ ]:
# Colab install — skip if already installed
# !pip install fawp-index[plot] -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import fawp_index
print(f'fawp-index v{fawp_index.__version__}')

## 1 · The Analytic Model

The core OATS model: Gaussian channel, linear noise growth.

$$I(\tau) = \frac{1}{2}\log_2\left(1 + \frac{P}{\sigma_0^2 + \alpha\tau}\right)$$

Agency horizon (closed form):

$$\tau_h = \max\left(0,\ \frac{P/(2^{2\varepsilon}-1) - \sigma_0^2}{\alpha}\right)$$

In [ ]:
from fawp_index.oats import AgencyHorizon
from fawp_index.explain import explain

ah = AgencyHorizon(
    P        = 1.0,    # signal power
    sigma0_sq= 0.01,   # base noise variance
    alpha    = 0.001,  # noise growth rate
    epsilon  = 0.1,    # MI threshold (bits)
)

result = ah.compute()
print(result.summary())

In [ ]:
print(explain(result))

In [ ]:
result.plot(show=True)

## 2 · E1: Parameter Sweep — Analytic vs Monte Carlo

Sweeps (P, α, ε) space and compares closed-form τ_h to Monte Carlo estimates.
Mean relative error for COVERED runs should be < 10%.

In [ ]:
# Load published E1 sweep data and compare
sweep = ah.compare_e1()
print(sweep.summary())

In [ ]:
sweep.plot_scaling(show=True)

In [ ]:
# Run your own sweep
my_sweep = ah.sweep(
    P_values      = [0.1, 1.0, 10.0, 100.0],
    alpha_values  = [0.0001, 0.001, 0.01],
    epsilon_values= [0.01, 0.1, 0.5],
)
print(my_sweep.summary())
my_sweep.dataframe().head(10)

## 3 · E2: Convergence — MI Estimator Follows 1/√N

Verifies the Monte Carlo estimator obeys Central Limit Theorem scaling.

In [ ]:
from fawp_index.data import E2_CONVERGENCE

df = pd.read_csv(E2_CONVERGENCE)
print(f'Sample sizes: {df["N"].min():,} → {df["N"].max():,}')
print(f'Final 95% CI width: {df["ci_width"].iloc[-1]:.6f} bits')

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(df['N'], df['ci_width'], 'bo-', lw=2, ms=5, label='95% CI width')
scale = df['ci_width'].iloc[0] * np.sqrt(df['N'].iloc[0])
ax.loglog(df['N'], scale / np.sqrt(df['N']), 'k--', lw=1.5, label='1/√N reference')
ax.set_xlabel('Sample size N'); ax.set_ylabel('CI width (bits)')
ax.set_title('E2: MI Estimator Convergence'); ax.legend(); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

## 4 · E3: Noise-Law Discrimination

Three noise growth models compared:
- **Linear**: σ²(τ) = σ₀² + α·τ  
- **Quadratic**: σ²(τ) = σ₀² + α·τ²  
- **Saturating**: σ²(τ) = σ_max · (1 − e^{−τ/τ_s})

In [ ]:
from fawp_index.data import E3_CURVES_LINEAR, E3_CURVES_QUADRATIC, E3_CURVES_SATURATING, E3_HORIZONS_SUMMARY

print('Horizons summary:')
print(pd.read_csv(E3_HORIZONS_SUMMARY).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
for path, label, color in [
    (E3_CURVES_LINEAR,     'Linear',     'darkorange'),
    (E3_CURVES_QUADRATIC,  'Quadratic',  'steelblue'),
    (E3_CURVES_SATURATING, 'Saturating', 'green'),
]:
    df = pd.read_csv(path)
    ax.plot(df['tau_s'], df['mi_mc_mean'], lw=2.2, color=color, label=label)
    ax.fill_between(df['tau_s'], df['mi_mc_ci95_low'], df['mi_mc_ci95_high'], alpha=0.12, color=color)
    if 'mi_analytic' in df.columns:
        ax.plot(df['tau_s'], df['mi_analytic'], '--', lw=1.2, color=color, alpha=0.5)

ax.set_xlabel('Latency τ'); ax.set_ylabel('MI (bits)')
ax.set_title('E3: Noise-Law Discrimination — MC mean ± 95% CI + analytic')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)
plt.tight_layout(); plt.show()

## 5 · E4: Distributional Robustness

The Agency Horizon is not an artifact of Gaussian assumptions.
Same decay behaviour under Uniform actions and Student-t noise.

In [ ]:
from fawp_index.oats import DistributionalRobustness

rob = DistributionalRobustness.from_e4_data()
print(rob.summary())
rob.plot(show=True)

## 6 · E5: Control Cliff

MI decays smoothly — but system stability collapses abruptly.
This makes the agency horizon *operational*, not just theoretical.

In [ ]:
from fawp_index.simulate import ControlCliff

cc = ControlCliff.from_e5_data()
print(cc.summary())
print()
print(explain(cc))
cc.plot(show=True)

## 7 · E6: Agency Capacity Surfaces

**Surface A**: Quantization cliff — agency collapses below 3-bit depth.  
**Surface B**: Erasure slope — 50% packet loss ≈ halves the effective horizon.

In [ ]:
from fawp_index.capacity import CapacitySurface

cs = CapacitySurface.from_e6_data()
print(cs.summary())
cs.plot(surface='both', show=True)

In [ ]:
# Surface A only — quantization cliff
cs.plot(surface='A', show=True)

In [ ]:
# Surface B only — erasure slope
cs.plot(surface='B', show=True)

## 8 · E7: Quantum Consistency Checks

Sanity check: the pipeline respects the difference between **correlation** and **communication**.
- Bell/CHSH violations ✓ (non-classical correlation)
- No-signaling preserved ✓ (Δ_NS ≈ machine precision)

In [ ]:
from fawp_index.data import E7_QUANTUM

df = pd.read_csv(E7_QUANTUM)
print(f'Max CHSH S:          {df["CHSH"].max():.4f}  (Tsirelson bound: {2*np.sqrt(2):.4f})')
print(f'Max NS deviation:    {df["NS_Dev"].max():.2e}  (should be ≈ machine precision)')
print(f'Bell threshold v=1/√2 ≈ {1/np.sqrt(2):.4f} — exceeded: {(df["CHSH"] > 2).sum()} values')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(df['v'], df['CHSH'], 'bo-', lw=2, ms=4, label='Simulated S')
ax1.plot(df['v'], df['CHSH_theory'], 'k--', lw=1.5, label='Theory: 2√2·v')
ax1.axhline(2, color='red', ls=':', lw=1.5, alpha=0.7, label='Classical bound (S=2)')
ax1.axhline(2*np.sqrt(2), color='green', ls=':', lw=1.5, alpha=0.7, label='Tsirelson (2√2)')
ax1.set_xlabel('Visibility v'); ax1.set_ylabel('CHSH parameter S')
ax1.set_title('E7a: Bell/CHSH Test'); ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

ns = df['NS_Dev'].replace(0, 1e-20)
ax2.semilogy(df['v'], ns, 'r^-', lw=2, ms=5, label='No-signaling deviation')
ax2.axhline(1e-15, color='gray', ls='--', lw=1.2, label='Machine precision')
ax2.set_xlabel('Visibility v'); ax2.set_ylabel('Δ_NS')
ax2.set_title('E7b: No-Signaling Check'); ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout(); plt.show()

## 9 · Plain-English Diagnosis with explain()

`explain()` auto-detects the result type and returns a human-readable narrative.

In [ ]:
# Explain any OATS result
from fawp_index.oats import AgencyHorizon
from fawp_index.explain import explain

# Short horizon — severe case
result_severe = AgencyHorizon(P=0.1, sigma0_sq=0.05, alpha=0.05, epsilon=0.1).compute()
print(explain(result_severe))

In [ ]:
# Explain a control cliff
from fawp_index.simulate import ControlCliff
print(explain(ControlCliff.from_e5_data()))

In [ ]:
# Explain a full FAWP result
from fawp_index import FAWPAlphaIndex

rng = np.random.default_rng(42)
pred   = rng.normal(0, 1, 5000)
future = pred[20:] + rng.normal(0, 0.3, 4980)
action = rng.normal(0, 0.001, 4980)   # near zero = FAWP
obs    = rng.normal(0, 0.1, 4980)

fawp_result = FAWPAlphaIndex().compute(pred[:4980], future, action, obs)
print(explain(fawp_result))

---

## Citation

```bibtex
@article{clayton2026agency,
  author = {Ralph Clayton},
  title  = {Forecasting Without Power: Agency Horizons and the Leverage Gap},
  year   = {2026},
  doi    = {10.5281/zenodo.18663547}
}
```

📗 [*Forecasting Without Power* on Amazon](https://www.amazon.com/dp/B0GS1ZVNM7/)  
📦 [fawp-index on PyPI](https://pypi.org/project/fawp-index/)  
📂 [GitHub](https://github.com/DrRalphClayton/fawp-index)